# FedBiEF-Prox on MNIST: single Top-K run (fast version)

**What this notebook does.** It runs **one** federated training run with **one** Top-K value (set `TOP_K_UP` / `TOP_K_DOWN` in CONFIG), one seed, and produces the training-loss curve, accuracy curves, communication cost, error-buffer diagnostics, confusion matrix and a summary table. It is the short version of the sweep notebook, built to finish quickly.

**Why it is faster.**
- one K and one seed instead of 6 K values x 3 seeds
- local steps update the parameters in place (no flatten/copy at every step)
- during training the global model is evaluated every `EVAL_EVERY` rounds on fixed random **subsets** of validation and test (`EVAL_SUBSET` images each); the **full** validation and test sets are used once, at the end
- the optional extras (uncompressed reference run, centralized baseline) are switched off by default

**Setup (same as before).** 20 clients with a Dirichlet non-IID split; each round 5 clients (`C_FRAC = 0.25`) are sampled uniformly without replacement, so every client has equal probability. FL is simulated on one machine.

**Notation.** `K_LOCAL` is the number of local steps (the algorithm's K). Top-K means the fraction of entries KEPT: 0.10 keeps 10%, which is 90% sparsity.

### Algorithm implemented (Algorithm 2)

```
Init: client error buffers e^(i) = 0 for all i;  server buffer e_s = 0
for t = 1..T:
    sample S_t subset of [N]  (uniform, without replacement)
    for each client i in S_t:
        w_{t,0}^(i) = w_t
        for k = 0..K_LOCAL-1:
            w_{t,k+1}^(i) = w_{t,k}^(i) - eta * ( g + lambda * (w_{t,k}^(i) - w_t) )
        Delta^(i) = w_t - w_{t,K_LOCAL}^(i)
        p^(i)     = Delta^(i) + e^(i)
        send Delta_tilde^(i) = C_c(p^(i))            # uplink Top-K
        e^(i)     = p^(i) - Delta_tilde^(i)
    clients not in S_t: e^(i) unchanged
    Server:
        Delta_bar = mean of Delta_tilde^(i) over S_t
        q         = Delta_bar + e_s
        Omega     = C_s(q)                           # downlink Top-K
        e_s       = q - Omega
        w_{t+1}   = w_t - eta_s * Omega
```

**Sign convention.** `Delta = w_t - w_local` is a pseudo-gradient, so the server subtracts `Omega`. With Top-K = 100%, `eta_s = 1` and `lambda = 0` this is plain FedAvg (a unit test checks it).

In [ ]:
import os, sys, gzip, json, math, time, hashlib, random, struct
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from IPython.display import display, Markdown

CONFIG = dict(
    # ---------------- data ----------------
    DATA_ROOT = os.environ.get("FL_DATA_ROOT", r"E:\Federated learning"),
    ALLOW_DOWNLOAD = True,          # if MNIST is not found under DATA_ROOT, download it there
    VAL_FRACTION = 0.1,
    # ---------------- model ----------------
    MODEL_NAME = "cnn",             # "cnn" or "mlp" (no BatchNorm)
    # ---------------- federated setup ----------------
    N_CLIENTS = 20,
    C_FRAC = 0.25,                  # 0.25 * 20 = 5 clients per round (partial participation)
    T_ROUNDS = 50,
    K_LOCAL = 20,                   # exact number of local minibatch steps
    BATCH_SIZE = 32,
    ETA_LOCAL = 0.05,               # eta
    ETA_GLOBAL = 1.0,               # eta_s
    PROX_LAMBDA = 0.01,             # lambda
    DIRICHLET_ALPHA = 0.5,
    MIN_SAMPLES_PER_CLIENT = 20,
    # ---------------- compression: THE ONE K VALUE (fraction KEPT) ----------------
    TOP_K_UP = 0.10,                # uplink  (client -> server)
    TOP_K_DOWN = 0.10,              # downlink (server -> clients)
    TOPK_MODE = "per_layer",        # "per_layer" or "global"
    # ---------------- run control ----------------
    SEED = 0,
    EVAL_EVERY = 2,                 # evaluate the global model every this many rounds (on subsets)
    EVAL_SUBSET = 2000,             # images in each monitoring subset (val and test)
    RUN_UNCOMPRESSED_REF = False,   # True adds a second run with K=100% for comparison (about 2x the time)
    RUN_CENTRALIZED = False,        # True adds a centralized baseline with the same number of gradient steps
    FORCE_RERUN = False,            # True ignores saved results and reruns
    RESULTS_DIR = "results_mnist_single_k",
    QUICK_TEST = os.environ.get("FL_QUICK", "0") == "1",
)
if CONFIG["QUICK_TEST"]:
    CONFIG.update(T_ROUNDS=4, K_LOCAL=3, EVAL_EVERY=1, RESULTS_DIR="results_mnist_single_k_quick")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| torch", torch.__version__, "| threads:", torch.get_num_threads())

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

N_PER_ROUND = max(1, round(CONFIG["C_FRAC"] * CONFIG["N_CLIENTS"]))
print(f"{CONFIG['N_CLIENTS']} clients, {N_PER_ROUND} sampled per round, {CONFIG['T_ROUNDS']} rounds, "
      f"{CONFIG['K_LOCAL']} local steps, Top-K up/down = {CONFIG['TOP_K_UP']}/{CONFIG['TOP_K_DOWN']}")

## 1. MNIST loading (from `DATA_ROOT`)

The loader looks inside `DATA_ROOT` (up to 4 folder levels deep) for, in this order: (1) the standard IDX files (`train-images-idx3-ubyte` and friends, plain or `.gz`), (2) `mnist_train.csv` / `mnist_test.csv`. If neither exists and `ALLOW_DOWNLOAD` is true, it downloads MNIST into `DATA_ROOT` with torchvision. It never creates synthetic data: if nothing can be loaded, it raises an error.

In [ ]:
def _read_idx(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rb") as f:
        data = f.read()
    magic = struct.unpack(">I", data[:4])[0]
    ndim = magic & 0xFF
    dims = struct.unpack(">" + "I" * ndim, data[4:4 + 4 * ndim])
    return np.frombuffer(data, dtype=np.uint8, offset=4 + 4 * ndim).reshape(dims)

IDX_NAMES = {"train-images-idx3-ubyte": "xtr", "train-labels-idx1-ubyte": "ytr",
             "t10k-images-idx3-ubyte": "xte", "t10k-labels-idx1-ubyte": "yte"}

def _walk(root, max_depth=4):
    root = str(root); base = root.rstrip("\\/").count(os.sep)
    for dp, dn, fn in os.walk(root):
        if dp.count(os.sep) - base >= max_depth:
            dn[:] = []
        for f in fn:
            yield Path(dp) / f

def _find_idx(root):
    found = {}
    for p in _walk(root):
        name = p.name.lower().replace(".", "-")
        if name.endswith("-gz"): name = name[:-3]
        if name in IDX_NAMES and IDX_NAMES[name] not in found:
            found[IDX_NAMES[name]] = p
    return found if len(found) == 4 else None

def _find_csv(root):
    tr = te = None
    for p in _walk(root):
        n = p.name.lower()
        if n == "mnist_train.csv" and tr is None: tr = p
        if n == "mnist_test.csv" and te is None: te = p
    return (tr, te) if tr and te else None

def load_mnist(root, allow_download=True):
    root = Path(root)
    if root.exists():
        idx = _find_idx(root)
        if idx:
            print("MNIST source: IDX files ->", {k: str(v) for k, v in idx.items()})
            return (_read_idx(idx["xtr"]), _read_idx(idx["ytr"]),
                    _read_idx(idx["xte"]), _read_idx(idx["yte"]))
        csv = _find_csv(root)
        if csv:
            print("MNIST source: CSV ->", csv[0], csv[1])
            a = pd.read_csv(csv[0], header=None).values.astype(np.uint8)
            b = pd.read_csv(csv[1], header=None).values.astype(np.uint8)
            return (a[:, 1:].reshape(-1, 28, 28), a[:, 0], b[:, 1:].reshape(-1, 28, 28), b[:, 0])
    if allow_download:
        print("MNIST not found under", root, "-> downloading with torchvision into that folder")
        root.mkdir(parents=True, exist_ok=True)
        from torchvision import datasets
        tr = datasets.MNIST(str(root), train=True, download=True)
        te = datasets.MNIST(str(root), train=False, download=True)
        return (tr.data.numpy(), tr.targets.numpy(), te.data.numpy(), te.targets.numpy())
    raise FileNotFoundError(f"No MNIST data found under {root} and ALLOW_DOWNLOAD is False.")

xtr_raw, ytr_raw, xte_raw, yte_raw = load_mnist(CONFIG["DATA_ROOT"], CONFIG["ALLOW_DOWNLOAD"])
print("raw shapes:", xtr_raw.shape, ytr_raw.shape, xte_raw.shape, yte_raw.shape)

# ---- stratified validation hold-out (fixed seed, identical for every run) ----
def stratified_split(labels, frac, seed=0):
    rng = np.random.RandomState(seed)
    val_idx = []
    for c in range(10):
        ids = np.where(labels == c)[0]
        rng.shuffle(ids)
        val_idx.extend(ids[: int(round(frac * len(ids)))].tolist())
    val_idx = np.array(sorted(val_idx))
    mask = np.ones(len(labels), bool); mask[val_idx] = False
    return np.where(mask)[0], val_idx

tr_idx, va_idx = stratified_split(ytr_raw, CONFIG["VAL_FRACTION"])

def to_tensor(x):   # uint8 (N,28,28) -> normalized float (N,1,28,28)
    t = torch.from_numpy(x.astype(np.float32) / 255.0)
    return ((t - 0.1307) / 0.3081).unsqueeze(1).contiguous()

X_TRAIN = to_tensor(xtr_raw[tr_idx]).to(DEVICE); Y_TRAIN = torch.from_numpy(ytr_raw[tr_idx].astype(np.int64)).to(DEVICE)
X_VAL   = to_tensor(xtr_raw[va_idx]).to(DEVICE); Y_VAL   = torch.from_numpy(ytr_raw[va_idx].astype(np.int64)).to(DEVICE)
X_TEST  = to_tensor(xte_raw).to(DEVICE);         Y_TEST  = torch.from_numpy(yte_raw.astype(np.int64)).to(DEVICE)

assert len(Y_TRAIN) > 0 and len(Y_VAL) > 0 and len(Y_TEST) > 0, "an MNIST split is empty"
print(f"train={len(Y_TRAIN)}  val={len(Y_VAL)}  test={len(Y_TEST)}")
display(pd.DataFrame({"train": np.bincount(Y_TRAIN.cpu().numpy(), minlength=10),
                      "val":   np.bincount(Y_VAL.cpu().numpy(), minlength=10),
                      "test":  np.bincount(Y_TEST.cpu().numpy(), minlength=10)}).rename_axis("digit").T)

## 2. Heterogeneous (non-IID) client partition

The TRAIN portion (after removing validation) is split across the clients with a Dirichlet(alpha) distribution over the 10 digit labels. Smaller alpha gives more skewed clients. The partition is computed **once** with a fixed seed and reused for every Top-K value and every seed. Validation and test sets are global and are never partitioned.

In [ ]:
def dirichlet_partition(labels, n_clients, alpha, min_size, seed):
    rng = np.random.RandomState(seed)
    for attempt in range(2000):
        batches = [[] for _ in range(n_clients)]
        for c in range(10):
            ids = np.where(labels == c)[0]
            rng.shuffle(ids)
            props = rng.dirichlet(np.repeat(alpha, n_clients))
            cuts = (np.cumsum(props) * len(ids)).astype(int)[:-1]
            for cid, part in enumerate(np.split(ids, cuts)):
                batches[cid].extend(part.tolist())
        if min(len(b) for b in batches) >= min_size:
            return [np.array(b, dtype=np.int64) for b in batches], attempt + 1
    raise RuntimeError("Could not satisfy MIN_SAMPLES_PER_CLIENT; raise alpha or lower the minimum.")

parts, tries = dirichlet_partition(Y_TRAIN.cpu().numpy(), CONFIG["N_CLIENTS"], CONFIG["DIRICHLET_ALPHA"],
                                   CONFIG["MIN_SAMPLES_PER_CLIENT"], seed=CONFIG["SEED"])
CLIENT_IDX = [torch.from_numpy(p) for p in parts]          # CPU index tensors
assert sum(len(p) for p in parts) == len(Y_TRAIN) and len(set(np.concatenate(parts).tolist())) == len(Y_TRAIN)
print(f"Dirichlet(alpha={CONFIG['DIRICHLET_ALPHA']}) partition found after {tries} attempt(s). "
      f"Client sizes: min={min(len(p) for p in parts)}, max={max(len(p) for p in parts)}")

counts = np.zeros((CONFIG["N_CLIENTS"], 10), dtype=int)
ylab = Y_TRAIN.cpu().numpy()
for cid, p in enumerate(parts):
    counts[cid] = np.bincount(ylab[p], minlength=10)

fig, ax = plt.subplots(figsize=(8, 0.28 * CONFIG["N_CLIENTS"] + 2))
im = ax.imshow(counts, aspect="auto", cmap="viridis")
ax.set_xlabel("Digit label"); ax.set_ylabel("Client ID")
ax.set_xticks(range(10)); ax.set_yticks(range(CONFIG["N_CLIENTS"]))
ax.set_title(f"Per-client digit counts: Dirichlet(alpha={CONFIG['DIRICHLET_ALPHA']}) non-IID split")
plt.colorbar(im, ax=ax, label="samples"); plt.tight_layout()
os.makedirs(os.path.join(CONFIG["RESULTS_DIR"], "plots"), exist_ok=True)
plt.savefig(os.path.join(CONFIG["RESULTS_DIR"], "plots", "client_heatmap.png"), dpi=130); plt.show()

## 3. Model

Two options, both with **no BatchNorm**, so every entry of the parameter vector is a learned weight and the compressor never touches running statistics.
All federated logic works on a single flat parameter vector `w`, and `LAYERS` records where each tensor sits inside it (used by the per-layer compressor).

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 32, 3); self.c2 = nn.Conv2d(32, 64, 3)
        self.f1 = nn.Linear(64 * 5 * 5, 128); self.f2 = nn.Linear(128, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.c1(x)), 2)      # 28 -> 26 -> 13
        x = F.max_pool2d(F.relu(self.c2(x)), 2)      # 13 -> 11 -> 5
        return self.f2(F.relu(self.f1(x.flatten(1))))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.f1 = nn.Linear(784, 256); self.f2 = nn.Linear(256, 128); self.f3 = nn.Linear(128, 10)
    def forward(self, x):
        x = x.flatten(1)
        return self.f3(F.relu(self.f2(F.relu(self.f1(x)))))

def build_model(name=None):
    name = name or CONFIG["MODEL_NAME"]
    return {"cnn": CNN, "mlp": MLP}[name]().to(DEVICE)

def get_flat(model):      return torch.cat([p.data.reshape(-1) for p in model.parameters()])
def get_flat_grad(model): return torch.cat([p.grad.reshape(-1) for p in model.parameters()])
def set_flat(model, vec):
    o = 0
    for p in model.parameters():
        n = p.numel(); p.data.copy_(vec[o:o + n].view_as(p)); o += n

_m = build_model()
LAYERS = [(n, p.numel()) for n, p in _m.named_parameters()]
N_PARAMS = sum(n for _, n in LAYERS)
DENSE_BYTES = N_PARAMS * 4
assert not any(isinstance(m, nn.modules.batchnorm._BatchNorm) for m in _m.modules())
print(f"Model '{CONFIG['MODEL_NAME']}': {N_PARAMS:,} parameters; dense update = {DENSE_BYTES/1e6:.3f} MB (float32)")
del _m

## 4. Top-K compressor and communication accounting

`top_k_compressor(vec, k_fraction)` keeps the `k_fraction` of entries with the **largest magnitude** and zeroes the rest. `k_fraction` is the fraction KEPT (0.1 keeps 10%, which is 90% sparsity).

- `per_layer` (default): the top fraction is taken inside every tensor, so small layers such as biases are not crowded out by big ones.
- `global`: one threshold over the whole vector, so large-magnitude layers dominate.
- Indices come from `torch.topk`, so ties can never keep more than `k` entries.
- `k_fraction >= 1.0` skips masking entirely (uncompressed reference).

One compressor function is used for both directions (`C_c` and `C_s` differ only in the `k_fraction` passed in).

**Bytes per transmission** are `min(dense, nnz * 4 + nnz * 4)` (float32 value plus int32 index) for `k_fraction < 1`, and `dense` for `k_fraction = 1`. Uplink is summed over selected clients, and the downlink counts one broadcast of `Omega` per selected client. 1 MB = 10^6 bytes.

In [ ]:
def top_k_compressor(vec, k_fraction, mode=None):
    '''Keep the k_fraction (KEPT fraction, not sparsity) largest-magnitude entries. Returns (compressed, nnz).'''
    mode = mode or CONFIG["TOPK_MODE"]
    if k_fraction >= 1.0:
        return vec, vec.numel()
    out = torch.zeros_like(vec); nnz = 0
    if mode == "global":
        k = max(1, int(round(k_fraction * vec.numel())))
        idx = torch.topk(vec.abs(), k, sorted=False).indices
        out[idx] = vec[idx]; nnz = k
    elif mode == "per_layer":
        o = 0
        for _, n in LAYERS:
            seg = vec[o:o + n]
            k = max(1, int(round(k_fraction * n)))
            idx = torch.topk(seg.abs(), k, sorted=False).indices
            out[o:o + n][idx] = seg[idx]; nnz += k; o += n
    else:
        raise ValueError(mode)
    return out, nnz

def tx_bytes(nnz, k_fraction):
    return DENSE_BYTES if k_fraction >= 1.0 else min(DENSE_BYTES, nnz * 8)

## 5. Local training (K_LOCAL minibatch steps with the proximal term)

Each selected client starts from the global weights `w_t` and takes **exactly `K_LOCAL` minibatch steps** from a cycling minibatch iterator (a client with few batches wraps around; this is not `K_LOCAL` epochs).

Each step applies the update directly to the parameters, in place:

`w <- w - eta * ( g + lambda * (w - w_t) )`

where `g` is the cross-entropy gradient only; the proximal pull is added explicitly (it equals the gradient of `(lambda/2) * ||w - w_t||^2`). Compared with the sweep notebook, nothing is flattened or copied inside the step loop, and the proximal value is computed once at the end of local training instead of at every step. The logged training loss is the data loss only.

In [ ]:
class BatchCycler:
    '''Infinite minibatch iterator over a fixed index set; reshuffles when it runs out.'''
    def __init__(self, idx, bs, gen):
        self.idx, self.bs, self.gen = idx, min(bs, len(idx)), gen
        self._shuffle()
    def _shuffle(self):
        self.perm = self.idx[torch.randperm(len(self.idx), generator=self.gen)]; self.ptr = 0
    def next(self):
        if self.ptr + self.bs > len(self.perm): self._shuffle()
        b = self.perm[self.ptr:self.ptr + self.bs]; self.ptr += self.bs
        return b.to(DEVICE)

def local_update(model, w_t, client_idx, gen):
    '''K_LOCAL minibatch steps from w_t, updating parameters in place. Returns (w_local, mean CE loss, proximal value at the end).'''
    cfg = CONFIG; eta, lam = cfg["ETA_LOCAL"], cfg["PROX_LAMBDA"]
    set_flat(model, w_t); model.train()
    params = list(model.parameters())
    w0, o = [], 0                                   # views of the frozen global weights, one per tensor
    for p in params:
        n = p.numel(); w0.append(w_t[o:o + n].view_as(p)); o += n
    cyc = BatchCycler(client_idx, cfg["BATCH_SIZE"], gen)
    losses = []
    for _ in range(cfg["K_LOCAL"]):
        b = cyc.next()
        for p in params: p.grad = None
        loss = F.cross_entropy(model(X_TRAIN[b]), Y_TRAIN[b])
        loss.backward()
        with torch.no_grad():
            for p, g0 in zip(params, w0):           # w <- w - eta * (g + lambda * (w - w_t))
                p.add_(p.grad + lam * (p - g0), alpha=-eta)
        losses.append(loss.detach())
    w_loc = get_flat(model).clone()
    prox = 0.5 * lam * (w_loc - w_t).pow(2).sum().item()
    return w_loc, torch.stack(losses).mean().item(), prox

@torch.no_grad()
def evaluate(model, w, X, Y):
    set_flat(model, w); model.eval()
    tot, correct, preds = 0.0, 0, []
    for i in range(0, len(Y), 4096):
        out = model(X[i:i + 4096])
        tot += F.cross_entropy(out, Y[i:i + 4096], reduction="sum").item()
        p = out.argmax(1); preds.append(p); correct += (p == Y[i:i + 4096]).sum().item()
    preds = torch.cat(preds)
    conf = torch.bincount(Y * 10 + preds, minlength=100).view(10, 10).cpu().numpy()
    return tot / len(Y), correct / len(Y), conf

def server_aggregate(w_t, delta_tildes, e_s, k_down, eta_s):
    '''Server half of Algorithm 2. Returns (w_next, e_s_next, Omega, nnz_omega).'''
    delta_bar = torch.stack(delta_tildes).mean(0)        # unweighted mean over S_t
    q = delta_bar + e_s
    omega, nnz = top_k_compressor(q, k_down)             # downlink compression C_s
    return w_t - eta_s * omega, q - omega, omega, nnz

## 6. Unit tests (run before the experiment)

1. **Proximal gradient**: autograd on `CE + (lambda/2)||w - w_t||^2` equals the manual `g_CE + lambda (w - w_t)`. Weights are moved away from `w_t` first so the test cannot pass vacuously.
2. **Compressor**: exact number of kept entries, and kept entries are at least as large as dropped ones.
3. **FedAvg reduction**: with `k = 1`, `eta_s = 1` the new global model is the average of the local models.
4. **Sampling**: every round has exactly `m` distinct clients.
5. **In-place local update**: the optimized `local_update` (2 steps) matches a plain reference implementation of `w <- w - eta (g + lambda (w - w_t))`, including the proximal term.

In [ ]:
def make_schedule(seed):
    '''Client-selection schedule: uniform, equal probability, WITHOUT replacement; reused across all K.'''
    rng = random.Random(10_000_000 + seed)
    return [sorted(rng.sample(range(CONFIG["N_CLIENTS"]), N_PER_ROUND)) for _ in range(CONFIG["T_ROUNDS"])]

def unit_test_proximal_term():
    set_seed(0); model = build_model(); model.train(); lam = CONFIG["PROX_LAMBDA"]
    w_t = {n: p.detach().clone() for n, p in model.named_parameters()}
    with torch.no_grad():
        for p in model.parameters(): p.add_(0.05 * torch.randn_like(p))      # move away from w_t
    x = torch.randn(8, 1, 28, 28, device=DEVICE); y = torch.randint(0, 10, (8,), device=DEVICE)
    # 1) autograd path: CE + (lambda/2)*||w - w_t||^2
    model.zero_grad()
    ce = F.cross_entropy(model(x), y)
    prox = 0.5 * lam * sum(((p - w_t[n]) ** 2).sum() for n, p in model.named_parameters())
    (ce + prox).backward()
    auto = {n: p.grad.clone() for n, p in model.named_parameters()}
    # 2) manual path with a FRESH forward/backward: g_CE + lambda*(w - w_t)
    model.zero_grad()
    F.cross_entropy(model(x), y).backward()
    max_prox = 0.0
    for n, p in model.named_parameters():
        pg = lam * (p.detach() - w_t[n]); max_prox = max(max_prox, pg.abs().max().item())
        diff = (auto[n] - (p.grad + pg)).abs().max().item()
        assert diff < 1e-6, f"Proximal gradient mismatch in {n}: diff {diff}"
    assert max_prox > 0, "proximal contribution is zero, the test would be vacuous"
    print("Proximal term unit test passed.")

def unit_test_compressor():
    set_seed(1); v = torch.randn(N_PARAMS, device=DEVICE)
    for mode in ["per_layer", "global"]:
        for kf in [0.01, 0.1, 0.5]:
            out, nnz = top_k_compressor(v, kf, mode)
            assert int((out != 0).sum()) <= nnz
            spans = [(0, N_PARAMS)] if mode == "global" else None
            if spans is None:
                spans, o = [], 0
                for _, n in LAYERS: spans.append((o, o + n)); o += n
            tot = 0
            for a, b in spans:
                k = max(1, int(round(kf * (b - a)))); tot += k
                kept = out[a:b] != 0
                assert int(kept.sum()) == k, (mode, kf, int(kept.sum()), k)
                if int(kept.sum()) < (b - a):
                    assert v[a:b][kept].abs().min() >= v[a:b][~kept].abs().max()
            assert nnz == tot
    out, nnz = top_k_compressor(v, 1.0); assert out is v and nnz == N_PARAMS
    print("Compressor unit test passed.")

def unit_test_fedavg_reduction():
    set_seed(2); w_t = torch.randn(N_PARAMS, device=DEVICE)
    locals_ = [w_t + 0.1 * torch.randn_like(w_t) for _ in range(5)]
    deltas = [w_t - wl for wl in locals_]
    w_next, e_s, omega, _ = server_aggregate(w_t, deltas, torch.zeros_like(w_t), 1.0, 1.0)
    assert torch.allclose(w_next, torch.stack(locals_).mean(0), atol=1e-6) and float(e_s.abs().max()) == 0.0
    print("FedAvg reduction unit test passed.")

def unit_test_schedule():
    s = make_schedule(0)
    assert all(len(set(r)) == len(r) == N_PER_ROUND for r in s) and len(s) == CONFIG["T_ROUNDS"]
    assert make_schedule(0) == s
    print("Sampling unit test passed.")

def unit_test_local_update():
    set_seed(3); model = build_model(); w_t = get_flat(model).clone()
    old_k = CONFIG["K_LOCAL"]; CONFIG["K_LOCAL"] = 2
    try:
        idx = CLIENT_IDX[0]; eta, lam = CONFIG["ETA_LOCAL"], CONFIG["PROX_LAMBDA"]
        w_loc, _, _ = local_update(model, w_t, idx, torch.Generator().manual_seed(5))
        cyc = BatchCycler(idx, CONFIG["BATCH_SIZE"], torch.Generator().manual_seed(5)); w = w_t.clone()
        for _ in range(2):
            b = cyc.next(); set_flat(model, w); model.train(); model.zero_grad(set_to_none=True)
            F.cross_entropy(model(X_TRAIN[b]), Y_TRAIN[b]).backward()
            w = w - eta * (get_flat_grad(model) + lam * (w - w_t))
        assert torch.allclose(w_loc, w, atol=1e-5), float((w_loc - w).abs().max())
    finally:
        CONFIG["K_LOCAL"] = old_k
    print("In-place local update unit test passed.")

unit_test_proximal_term(); unit_test_compressor(); unit_test_fedavg_reduction(); unit_test_schedule(); unit_test_local_update()

## 7. One full FL run

`run_fl_experiment` builds a fresh model and zero error buffers, then follows Algorithm 2 for `T_ROUNDS` rounds.

- Logged every round: local training loss (data term), proximal value, uplink/downlink bytes, cumulative MB, sparsity per direction, the fraction of the update's L2 norm kept by each compression, and the client/server error-buffer norms.
- Every `EVAL_EVERY` rounds the global model is evaluated on fixed random **subsets** of validation and test (`EVAL_SUBSET` images each). At the end it is evaluated once on the **full** validation and test sets; those are the reported final numbers.
- Progress is printed during the run with an ETA, starting after round 1.
- Bytes per transmission are `min(dense, 8 bytes per kept entry)`: 4 bytes for the value plus 4 for the index. At the dense size nothing is saved, so Top-K above 50% saves no bytes under this encoding.

In [ ]:
def _norm(x): return x.norm().item()

# fixed monitoring subsets (same for every run)
_g = torch.Generator().manual_seed(123)
_vs = torch.randperm(len(Y_VAL), generator=_g)[:CONFIG["EVAL_SUBSET"]].to(DEVICE)
_ts = torch.randperm(len(Y_TEST), generator=_g)[:CONFIG["EVAL_SUBSET"]].to(DEVICE)
X_VAL_S, Y_VAL_S, X_TEST_S, Y_TEST_S = X_VAL[_vs], Y_VAL[_vs], X_TEST[_ts], Y_TEST[_ts]

def run_fl_experiment(k_up, k_down, seed, schedule, verbose=True):
    cfg = CONFIG; T = cfg["T_ROUNDS"]; N = cfg["N_CLIENTS"]; nan = float("nan")
    set_seed(seed)
    model = build_model(); w = get_flat(model).clone()
    client_err = [torch.zeros_like(w) for _ in range(N)]; e_s = torch.zeros_like(w); seen = set()
    gen = torch.Generator().manual_seed(10_000 + seed)
    sel_counts = [0] * N; byte_check = {}

    def eval_sub():
        vl, va, _ = evaluate(model, w, X_VAL_S, Y_VAL_S)
        tl, ta, _ = evaluate(model, w, X_TEST_S, Y_TEST_S)
        return vl, va, tl, ta

    vl, va, tl, ta = eval_sub()
    rows = [dict(round=0, train_loss=nan, train_prox=nan, val_loss=vl, val_acc=va, test_loss=tl, test_acc=ta,
                 up_mb=0.0, down_mb=0.0, cum_mb=0.0, up_sparsity=nan, down_sparsity=nan,
                 up_norm_kept=nan, down_norm_kept=nan, client_err_norm=0.0, server_err_norm=0.0)]
    cum, t_start, every, last_va = 0.0, time.time(), max(1, T // 10), nan
    for t in range(1, T + 1):
        sel = schedule[t - 1]
        tilde, up_bytes, up_nnz, losses, proxs, kept_up = [], 0, [], [], [], []
        for cid in sel:
            sel_counts[cid] += 1; seen.add(cid)
            w_loc, l, pr = local_update(model, w, CLIENT_IDX[cid], gen)
            losses.append(l); proxs.append(pr)
            delta = w - w_loc                                   # Delta = w_t - w_{t,K}
            p = delta + client_err[cid]                         # p = Delta + e
            d_tilde, nnz = top_k_compressor(p, k_up)            # uplink: C_c(p)
            client_err[cid] = p - d_tilde                       # e <- p - Delta_tilde (non-selected clients unchanged)
            tilde.append(d_tilde); up_nnz.append(nnz); up_bytes += tx_bytes(nnz, k_up)
            kept_up.append(_norm(d_tilde) / max(_norm(p), 1e-12))
        w, e_s, omega, nnz_o = server_aggregate(w, tilde, e_s, k_down, cfg["ETA_GLOBAL"])
        down_bytes = len(sel) * tx_bytes(nnz_o, k_down)
        cum += (up_bytes + down_bytes) / 1e6
        if t == 1:
            byte_check = dict(dense_per_tx=DENSE_BYTES, up_per_client=up_bytes / len(sel), down_per_client=down_bytes / len(sel))
        if t % cfg["EVAL_EVERY"] == 0 or t == T: vl, va, tl, ta = eval_sub(); last_va = va
        else: vl = va = tl = ta = nan
        rows.append(dict(round=t, train_loss=float(np.mean(losses)), train_prox=float(np.mean(proxs)),
                         val_loss=vl, val_acc=va, test_loss=tl, test_acc=ta,
                         up_mb=up_bytes / 1e6, down_mb=down_bytes / 1e6, cum_mb=cum,
                         up_sparsity=1 - float(np.mean(up_nnz)) / N_PARAMS, down_sparsity=1 - nnz_o / N_PARAMS,
                         up_norm_kept=float(np.mean(kept_up)), down_norm_kept=_norm(omega) / max(_norm(e_s + omega), 1e-12),
                         client_err_norm=float(np.mean([_norm(client_err[c]) for c in seen])), server_err_norm=_norm(e_s)))
        if verbose and (t == 1 or t % every == 0 or t == T):
            el = time.time() - t_start
            print(f"  round {t:>3}/{T} | train loss {np.mean(losses):.4f} | val acc (subset) {last_va:.4f} | "
                  f"elapsed {el:.0f}s | ETA {el / t * (T - t):.0f}s")
    # final evaluation on the FULL validation and test sets
    fvl, fva, _ = evaluate(model, w, X_VAL, Y_VAL)
    ftl, fta, conf = evaluate(model, w, X_TEST, Y_TEST)
    return dict(k_up=k_up, k_down=k_down, seed=seed, rows=rows, conf=conf.tolist(),
                final=dict(val_loss=fvl, val_acc=fva, test_loss=ftl, test_acc=fta),
                train_seconds=time.time() - t_start,
                schedule_hash=hashlib.md5(str(schedule).encode()).hexdigest()[:8],
                sel_counts=sel_counts, byte_check=byte_check)

def run_centralized(seed):
    '''Upper-bound reference: same model, pooled training data, plain SGD, same total number of gradient steps.'''
    cfg = CONFIG; steps = cfg["T_ROUNDS"] * N_PER_ROUND * cfg["K_LOCAL"]
    set_seed(seed); model = build_model(); gen = torch.Generator().manual_seed(20_000 + seed)
    cyc = BatchCycler(torch.arange(len(Y_TRAIN)), cfg["BATCH_SIZE"], gen)
    opt = torch.optim.SGD(model.parameters(), lr=cfg["ETA_LOCAL"]); model.train()
    for _ in range(steps):
        b = cyc.next(); opt.zero_grad(set_to_none=True)
        F.cross_entropy(model(X_TRAIN[b]), Y_TRAIN[b]).backward(); opt.step()
    w = get_flat(model).clone()
    vl, va, _ = evaluate(model, w, X_VAL, Y_VAL); tl, ta, _ = evaluate(model, w, X_TEST, Y_TEST)
    return dict(seed=seed, steps=steps, val_loss=vl, val_acc=va, test_loss=tl, test_acc=ta)

## 8. Run

The result is saved to `RESULTS_DIR` under a name that includes a hash of the configuration. Running this cell again with the same settings loads the saved result instantly; set `FORCE_RERUN = True` to recompute. Changing the Top-K value or any hyperparameter changes the hash, so stale results are never reused.

In [ ]:
os.makedirs(CONFIG["RESULTS_DIR"], exist_ok=True)
_HASH_SKIP = {"DATA_ROOT", "ALLOW_DOWNLOAD", "RUN_UNCOMPRESSED_REF", "RUN_CENTRALIZED", "FORCE_RERUN", "RESULTS_DIR", "QUICK_TEST"}
CFG_HASH = hashlib.md5(json.dumps({k: v for k, v in CONFIG.items() if k not in _HASH_SKIP}, sort_keys=True).encode()).hexdigest()[:8]
SEED = CONFIG["SEED"]; T = CONFIG["T_ROUNDS"]
SCHEDULE = make_schedule(SEED)

def _path(kind, *parts):
    return os.path.join(CONFIG["RESULTS_DIR"], f"{CFG_HASH}_{kind}_" + "_".join(str(p) for p in parts) + ".json")

MAIN = f"FedBiEF-Prox (Top-K up/down = {CONFIG['TOP_K_UP']*100:g}% / {CONFIG['TOP_K_DOWN']*100:g}%)"
REF = "Uncompressed reference (K=100%)"
RUNS = [(MAIN, CONFIG["TOP_K_UP"], CONFIG["TOP_K_DOWN"])]
if CONFIG["RUN_UNCOMPRESSED_REF"]: RUNS.append((REF, 1.0, 1.0))

results = {}
for name, ku, kd in RUNS:
    f = _path("fl", ku, kd, SEED)
    if os.path.exists(f) and not CONFIG["FORCE_RERUN"]:
        results[name] = json.load(open(f)); print(f"Loaded saved result for: {name}")
    else:
        print(f"Running: {name}")
        res = run_fl_experiment(ku, kd, SEED, SCHEDULE)
        json.dump(res, open(f, "w")); results[name] = res
        print(f"Finished in {res['train_seconds']/60:.1f} min | FULL test accuracy {res['final']['test_acc']*100:.2f}% | "
              f"total communication {res['rows'][-1]['cum_mb']:.1f} MB")

CENT = None
if CONFIG["RUN_CENTRALIZED"]:
    f = _path("cent", SEED)
    if os.path.exists(f) and not CONFIG["FORCE_RERUN"]: CENT = json.load(open(f))
    else:
        t0 = time.time(); CENT = run_centralized(SEED); json.dump(CENT, open(f, "w"))
        print(f"Centralized baseline finished in {(time.time()-t0)/60:.1f} min")

DF = pd.DataFrame([dict(run=name, **row) for name, r in results.items() for row in r["rows"]])
print("Rows logged:", len(DF))

## 9. Sanity checks

In [ ]:
main = results[MAIN]; k_up, k_down = CONFIG["TOP_K_UP"], CONFIG["TOP_K_DOWN"]
dm = DF[(DF.run == MAIN) & (DF["round"] >= 1)]
print(f"Round-0 validation accuracy (random init, expect about 10%): {DF[(DF.run == MAIN) & (DF['round'] == 0)]['val_acc'].iloc[0]:.4f}")
print(f"Achieved sparsity: uplink {dm.up_sparsity.mean():.4f} (expected {1-k_up:.4f}), downlink {dm.down_sparsity.mean():.4f} (expected {1-k_down:.4f})")
if k_up < 1.0: assert abs(dm.up_sparsity.mean() - (1 - k_up)) < 0.01
if k_down < 1.0: assert abs(dm.down_sparsity.mean() - (1 - k_down)) < 0.01
if REF in results:
    z = DF[DF.run == REF]; assert z.client_err_norm.max() == 0 and z.server_err_norm.max() == 0
    print("Uncompressed reference: error buffers are exactly zero.")
expected = N_PER_ROUND * T / CONFIG["N_CLIENTS"]
print(f"Selections per client over {T} rounds (expected about {expected:.1f}): {main['sel_counts']}")
b = main["byte_check"]
print(f"Round-1 bytes per client: dense {b['dense_per_tx']/1e6:.3f} MB | uplink {b['up_per_client']/1e6:.3f} MB "
      f"({b['up_per_client']/b['dense_per_tx']*100:.1f}% of dense) | downlink {b['down_per_client']/1e6:.3f} MB "
      f"({b['down_per_client']/b['dense_per_tx']*100:.1f}% of dense)")

## 10. Summary table

Final accuracy and loss are measured on the **full** validation and test sets. "Rounds to 95%" uses the monitoring test subset curve, so it has a resolution of `EVAL_EVERY` rounds. "Saving vs dense" compares total bytes with sending a dense update in both directions every round.

In [ ]:
cfg = CONFIG
display(pd.DataFrame({"Setting": [
    "FL algorithm", "Dataset", "Model", "Clients (N)", "Clients per round", "Sampling rule", "Heterogeneity",
    "Rounds (T)", "Local steps (K_LOCAL)", "Batch size", "eta (local) / eta_s (server) / lambda",
    "Top-K kept: uplink / downlink", "Top-K mode", "Seed", "Parameters / dense update"],
    "Value": [
    "FedBiEF-Prox (Algorithm 2: bidirectional compression + error feedback + proximal term)", "MNIST",
    f"{cfg['MODEL_NAME'].upper()} (no BatchNorm)", cfg["N_CLIENTS"], N_PER_ROUND,
    "Uniform equal-probability, without replacement", f"Dirichlet(alpha={cfg['DIRICHLET_ALPHA']}) non-IID",
    cfg["T_ROUNDS"], cfg["K_LOCAL"], cfg["BATCH_SIZE"], f"{cfg['ETA_LOCAL']} / {cfg['ETA_GLOBAL']} / {cfg['PROX_LAMBDA']}",
    f"{cfg['TOP_K_UP']*100:g}% / {cfg['TOP_K_DOWN']*100:g}%", cfg["TOPK_MODE"], cfg["SEED"],
    f"{N_PARAMS:,} / {DENSE_BYTES/1e6:.3f} MB"]}).set_index("Setting"))

DENSE_TOTAL_MB = 2 * N_PER_ROUND * T * DENSE_BYTES / 1e6

def rounds_to_95(name):
    g = DF[(DF.run == name) & (DF.test_acc >= 0.95)]
    return int(g["round"].iloc[0]) if len(g) else "not reached"

rows = []
for name, r in results.items():
    g = DF[DF.run == name]; tot = r["rows"][-1]["cum_mb"]
    rows.append({"Run": name, "Final test acc (%)": round(r["final"]["test_acc"] * 100, 2),
                 "Final test loss": round(r["final"]["test_loss"], 4),
                 "Final val acc (%)": round(r["final"]["val_acc"] * 100, 2),
                 "Best val acc, subset (%)": round(g["val_acc"].max() * 100, 2),
                 "Rounds to 95% test acc": rounds_to_95(name),
                 "Total comm (MB)": round(tot, 1), "Saving vs dense (%)": round((1 - tot / DENSE_TOTAL_MB) * 100, 1),
                 "Run time (min)": round(r["train_seconds"] / 60, 1)})
if CENT:
    rows.append({"Run": "Centralized (same gradient steps)", "Final test acc (%)": round(CENT["test_acc"] * 100, 2),
                 "Final test loss": round(CENT["test_loss"], 4), "Final val acc (%)": round(CENT["val_acc"] * 100, 2),
                 "Best val acc, subset (%)": "-", "Rounds to 95% test acc": "-", "Total comm (MB)": "-",
                 "Saving vs dense (%)": "-", "Run time (min)": "-"})
SUMMARY = pd.DataFrame(rows)
display(SUMMARY.style.hide(axis="index"))
SUMMARY.to_csv(os.path.join(CONFIG["RESULTS_DIR"], "summary_table.csv"), index=False)

## 11. Plots

Accuracy and loss curves use the monitoring subsets (points every `EVAL_EVERY` rounds); the exact final numbers on the full sets are in the summary table. All figures are saved to `RESULTS_DIR/plots`. The per-client digit heatmap was drawn in section 2.

In [ ]:
PLOT_DIR = os.path.join(CONFIG["RESULTS_DIR"], "plots"); os.makedirs(PLOT_DIR, exist_ok=True)
STYLE = {MAIN: dict(color="tab:blue", lw=2), REF: dict(color="tab:gray", lw=1.6, ls="--")}

def line(ax, metric, ylabel, title, logy=False):
    for name in results:
        g = DF[DF.run == name].dropna(subset=[metric])
        ax.plot(g["round"], g[metric], label=name, **STYLE.get(name, {}))
    ax.set_xlabel("Communication round"); ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(alpha=0.3)
    if logy: ax.set_yscale("log")

def save(name): plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, name + ".png"), dpi=130); plt.show()

# 1. Training loss vs round (linear and log) and validation loss
fig, axs = plt.subplots(1, 3, figsize=(18, 4.5))
line(axs[0], "train_loss", "Local train loss (CE, data term)", "Training loss vs round")
line(axs[1], "train_loss", "Local train loss (CE, log scale)", "Training loss vs round (log y)", logy=True)
line(axs[2], "val_loss", "Validation loss (subset)", "Validation loss vs round")
axs[0].legend(fontsize=8); save("01_loss")

# 2. Accuracy vs round: validation, test, and a zoom on the top of the curve
fig, axs = plt.subplots(1, 3, figsize=(18, 4.5))
line(axs[0], "val_acc", "Validation accuracy (subset)", "Validation accuracy vs round")
line(axs[1], "test_acc", "Test accuracy (subset)", "Test accuracy vs round")
line(axs[2], "test_acc", "Test accuracy (subset)", "Test accuracy, zoom on 90-100%"); axs[2].set_ylim(0.9, 1.0)
axs[0].legend(fontsize=8); save("02_accuracy")

In [ ]:
# 3. Communication: cumulative MB and the bytes saved relative to a dense run
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
line(axs[0], "cum_mb", "Cumulative communication (MB)", "Cumulative network transmission")
rr = np.arange(0, T + 1); axs[0].plot(rr, rr * 2 * N_PER_ROUND * DENSE_BYTES / 1e6, color="tab:red", ls=":", label="dense updates both ways")
axs[0].legend(fontsize=8)
for name in results:
    g = DF[(DF.run == name) & (DF["round"] >= 1)]
    axs[1].plot(g["round"], g["up_norm_kept"], label=f"{name}: uplink", **STYLE.get(name, {}))
    axs[1].plot(g["round"], g["down_norm_kept"], label=f"{name}: downlink", alpha=0.7, ls="-.")
axs[1].set_xlabel("Communication round"); axs[1].set_ylabel("Fraction of update L2 norm kept"); axs[1].set_ylim(0, 1.05)
axs[1].set_title("Signal surviving compression"); axs[1].grid(alpha=0.3); axs[1].legend(fontsize=7)
save("03_communication")

# 4. Error-feedback buffer norms
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
line(axs[0], "client_err_norm", "Mean client error-buffer L2 norm", "Client error buffers (clients seen so far)")
line(axs[1], "server_err_norm", "Server error-buffer L2 norm", "Server error buffer")
axs[0].legend(fontsize=8); save("04_error_buffers")

# 5. Confusion matrices (full test set, row-normalised)
names = list(results)
fig, axs = plt.subplots(1, len(names), figsize=(6.2 * len(names), 5.4)); axs = np.atleast_1d(axs)
for ax, name in zip(axs, names):
    cm = np.array(results[name]["conf"]); cmn = cm / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(10):
        for j in range(10):
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=6.5, color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xticks(range(10)); ax.set_yticks(range(10)); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{name}\nfull test accuracy {np.trace(cm)/cm.sum()*100:.2f}%", fontsize=9)
save("05_confusion")

## 12. Conclusions (computed from the results above)

In [ ]:
m = results[MAIN]; fin = m["final"]; tot = m["rows"][-1]["cum_mb"]
d = DF[(DF.run == MAIN) & (DF["round"] >= 1)]
first, last = d["train_loss"].iloc[0], d["train_loss"].iloc[-1]
L = [f"- **{MAIN}**: full test accuracy **{fin['test_acc']*100:.2f}%**, test loss {fin['test_loss']:.4f}, "
     f"after {T} rounds with {N_PER_ROUND} of {CONFIG['N_CLIENTS']} clients per round (Dirichlet alpha={CONFIG['DIRICHLET_ALPHA']}).",
     f"- Training loss fell from {first:.3f} (round 1) to {last:.3f} (round {T}).",
     f"- Communication: {tot:.1f} MB in total versus {DENSE_TOTAL_MB:.1f} MB for dense updates both ways, a saving of "
     f"**{(1 - tot / DENSE_TOTAL_MB) * 100:.1f}%** (8 bytes per kept entry: value plus index).",
     f"- Rounds to reach 95% test accuracy (subset curve): **{rounds_to_95(MAIN)}**."]
if REF in results:
    rf = results[REF]["final"]; gap = (rf["test_acc"] - fin["test_acc"]) * 100
    L.append(f"- Against the uncompressed reference ({rf['test_acc']*100:.2f}%): gap of **{gap:.2f} pp** "
             f"({'within 1 pp, compression cost almost no accuracy here' if abs(gap) <= 1 else 'larger than 1 pp, compression cost accuracy here'}). "
             f"A single seed cannot show whether a gap this size is real, so treat it as indicative.")
else:
    L.append("- No uncompressed reference was run, so the accuracy cost of compression is not measured here. Set `RUN_UNCOMPRESSED_REF = True` to add it.")
if CENT:
    L.append(f"- Centralized training on the pooled data with the same number of gradient steps reaches {CENT['test_acc']*100:.2f}%, "
             f"a gap of {(CENT['test_acc'] - fin['test_acc'])*100:.2f} pp to the federated run.")
if CONFIG["TOP_K_UP"] < 1.0 or CONFIG["TOP_K_DOWN"] < 1.0:
    s = d["server_err_norm"]; c = d["client_err_norm"]
    L.append(f"- Error buffers: the server buffer norm at the last round is {s.iloc[-1]/max(s.median(),1e-12):.2f}x its median over the run and the client "
             f"buffer {c.iloc[-1]/max(c.median(),1e-12):.2f}x (about 1 or below means bounded; well above 1 means still growing).")
L.append("- This is one seed, so differences of a few tenths of a percentage point are within run-to-run noise.")
display(Markdown("\n".join(L)))